# 三个坐标，要用几个独立的系数？

我们在三维坐标里研究平面 `W = {(s, t, s+t)}`。取 `u=(1,0,1)`、`v=(0,1,1)`，再取 `d=u+v=(1,1,2)`。先在纸上回答：三支全留着、只留两支、只留一支，哪些选择还能表示整个 W？

依赖是 NumPy、Matplotlib；ipywidgets 用于复选框和滑块。没有 widgets 时可以修改下面的函数调用。请从第一格开始顺序运行。图中的点是有限抽样，完整的范围要由坐标关系来判断。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product

u = np.array([1., 0., 1.])
v = np.array([0., 1., 1.])
d = u + v
vectors = {'u': u, 'v': v, 'd': d}
colors = {'u': '#0072B2', 'v': '#D55E00', 'd': '#8B5CA6'}
s, t = 1.5, 0.5
target = np.array([s, t, s + t])
print('s*u + t*v =', s * u + t * v)
print('target      =', target)
print('d - u - v   =', d - u - v)

## 先观察两支向量

左图画每个组合的前两个坐标，右图把同一批组合放回三维坐标。目标为 `(1.5,0.5,2)`。先预测：它在平面里吗？u 和 v 各取几倍才能得到它？

下面把画图和检查封装成函数。检查只用这三支向量的坐标关系：u、v 的系数是 s、t；u、d 的系数是 s-t、t；v、d 的系数是 t-s、s。先手算这些关系，再用输出核对。

In [ ]:
def show_space(use_u=True, use_v=True, use_d=False, s=1.5, t=0.5, offset=0.0):
    selected = [name for name, enabled in
                [('u', use_u), ('v', use_v), ('d', use_d)] if enabled]
    target = np.array([s, t, s + t + offset])
    coefficients = list(product(np.linspace(-1.5, 1.5, 9), repeat=len(selected)))
    if selected:
        samples = np.array(coefficients) @ np.stack([vectors[name] for name in selected])
    else:
        samples = np.zeros((1, 3))

    fig = plt.figure(figsize=(11, 5.2), dpi=160, layout='constrained')
    ax_xy = fig.add_subplot(1, 2, 1)
    ax_3d = fig.add_subplot(1, 2, 2, projection='3d')
    ax_xy.scatter(samples[:, 0], samples[:, 1], s=16, alpha=0.3,
                  color='#19856F', label='sampled combinations')
    ax_3d.scatter(*samples.T, s=12, alpha=0.3, color='#19856F')
    grid_s, grid_t = np.meshgrid(np.linspace(-3, 3, 7), np.linspace(-3, 3, 7))
    ax_3d.plot_wireframe(grid_s, grid_t, grid_s + grid_t,
                         color='#AAB6BE', alpha=0.35, linewidth=0.7)
    for name in selected:
        vector = vectors[name]
        ax_xy.quiver(0, 0, *vector[:2], angles='xy', scale_units='xy', scale=1,
                     color=colors[name], width=0.009, label=name)
        ax_3d.quiver(0, 0, 0, *vector, color=colors[name],
                     linewidth=2.5, arrow_length_ratio=0.12)
        ax_3d.text(*(1.15 * vector), name, color=colors[name], fontsize=12)
    ax_xy.scatter(*target[:2], marker='*', s=180, color='#B83F4E',
                  edgecolor='white', zorder=5, label='target projection')
    ax_3d.scatter(*target, marker='*', s=150, color='#B83F4E',
                  edgecolor='white', depthshade=False)
    ax_3d.text(*target, '  target', color='#B83F4E', fontsize=10)
    ax_xy.set(xlim=(-3.4, 3.4), ylim=(-3.4, 3.4), xlabel='x = s', ylabel='y = t',
              title='First two coordinates')
    ax_xy.set_aspect('equal', adjustable='box')
    ax_xy.axhline(0, color='#8D9AA4', linewidth=0.7)
    ax_xy.axvline(0, color='#8D9AA4', linewidth=0.7)
    ax_xy.grid(alpha=0.2)
    ax_xy.legend(loc='lower left', fontsize=9)
    ax_3d.set(xlim=(-3.4, 3.4), ylim=(-3.4, 3.4), zlim=(-6.8, 6.8),
              xlabel='x', ylabel='y', zlabel='z', title='Plane W: z = x + y')
    ax_3d.set_box_aspect((1, 1, 2))
    ax_3d.view_init(elev=24, azim=-125)
    plt.show()

    witness = None
    if np.isclose(offset, 0):
        if use_u and use_v:
            witness = np.array([s, t] + ([0.] if use_d else []))
        elif use_u and use_d:
            witness = np.array([s - t, t])
        elif use_v and use_d:
            witness = np.array([t - s, s])
        elif use_u and np.isclose(t, 0):
            witness = np.array([s])
        elif use_v and np.isclose(s, 0):
            witness = np.array([t])
        elif use_d and np.isclose(s, t):
            witness = np.array([s])
        elif not selected and np.allclose(target, 0):
            witness = np.array([])
    print('Selected vectors:', selected)
    print('Dimension of their span:', min(len(selected), 2))
    print('Can reach every vector in W?', len(selected) >= 2)
    print('Target:', target, '; z - x - y =', offset)
    if witness is None:
        print('This target cannot be formed with the selected vectors.')
    else:
        output = sum((c * vectors[name] for c, name in zip(witness, selected)), np.zeros(3))
        print('One choice of coefficients:', witness)
        print('Combination:', output, '; matches target?', np.allclose(output, target))

show_space()

## 一次只改变一个条件

1. 取消 v，只留下 u。先让 t=0，再改成 t=1。为什么第一次能到达，第二次不能？
2. 加回 v，保持 offset=0，任意改变 s、t。用 `(s,t,s+t)=s*u+t*v` 解释每次为何都能到达。
3. 加入 d。图上的点可能变多，抽样范围也可能扩大；这能证明张成空间扩大了吗？用 `d=u+v` 检查。再取消 v，验证 u、d 仍能表示所有 W 内的目标。
4. 三支都选上，把 offset 改为 1。目标的前两个坐标不变，但它还在 W 内吗？为什么所有组合都无法到达它？
5. 取消所有向量，再把 s、t、offset 全部设为 0。现在唯一能得到的结果是什么？

没有滑块时，改下面这行的参数即可。输出里的维数只针对这三支已经分析过的向量，依据是任意两支线性无关、第三支由它们表示。它不是从散点数量推测出的结论。

In [ ]:
show_space(use_u=True, use_v=False, use_d=False, s=1.5, t=0.5, offset=0)

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    print('For controls, install ipywidgets; editing the function call above also works.')
else:
    display(widgets.interactive(
        show_space,
        use_u=widgets.Checkbox(value=True, description='u'),
        use_v=widgets.Checkbox(value=True, description='v'),
        use_d=widgets.Checkbox(value=False, description='d = u + v'),
        s=widgets.FloatSlider(value=1.5, min=-2, max=2, step=0.25, description='s'),
        t=widgets.FloatSlider(value=0.5, min=-2, max=2, step=0.25, description='t'),
        offset=widgets.FloatSlider(value=0, min=-1.5, max=1.5, step=0.25, description='z offset')))

## 离开图，自己说清楚

为什么 u、v、d 有三个坐标，却只张成二维空间？请验证 u、d 也是 W 的一组基。再用一句坐标关系说明：无论选择哪几支，为什么都不能得到 `(0,0,1)`？

图上的有限抽样只能帮忙发现现象。要说明所有目标是否可达，需要写出适用于任意 s、t 的系数，或找出每个组合都必须满足的关系。